In [52]:
import requests
import json
from pathlib import Path
import pandas as pd
import os
import json
import psycopg2
from datetime import date, datetime
from getpass import getpass
from psycopg2 import sql
from psycopg2.extras import Json

In [ ]:
def extract_data(url_start, file_name, hasfields=True):
    # Extract data from the API
    url = url_start

    attribute_data_url = []
    attribute_data = []

    while url:  # Continue while there is a next page
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if hasfields:
                attribute_data_url.extend(item["href"] for item in data["content"]["fields"])
            else:
                attribute_data_url.extend(item["href"] for item in data["content"])
            url = data.get("pageable", {}).get("nextPage")  # Update URL to the next page
        else:
            print(f"Error: {response.status_code}")
            break

    for url_data in attribute_data_url:
        response = requests.get(url_data)
        if response.status_code == 200:
            data = response.json()
            attribute_data.append(data)
        else:
            print(f"Error: {response.status_code} for URL: {url_data}")

    try:
        output_path = Path(f"data/raw/{file_name}.json")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", encoding="utf-8") as f:
            json.dump(attribute_data, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
extract_data("https://digi-api.com/api/v1/attribute", "attributes")
extract_data("https://digi-api.com/api/v1/field", "fields")
extract_data("https://digi-api.com/api/v1/level", "levels")
extract_data("https://digi-api.com/api/v1/type", "types")
extract_data("https://digi-api.com/api/v1/skill", "skills")
extract_data("https://digi-api.com/api/v1/digimon", "digimons", False)

In [25]:
def extract_field_values(obj, field, key="id"):
    data = obj.get(field)

    if isinstance(data, list):
        return [
            item.get(key)
            for item in data
            if isinstance(item, dict) and item.get(key) is not None
        ]

    if isinstance(data, dict):
        value = data.get(key)
        return [value] if value is not None else []

    return []

In [36]:
def transform_data():
    try:
        with open(f"data/raw/digimons.json", "r", encoding="utf-8") as f:
            digimon_list = json.load(f)
        
        digimons = []
        for digimon in digimon_list:
            digimons.append({
                "id": digimon["id"],
                "name": digimon["name"],
                "images": digimon["images"],
                "releaseDate": digimon["releaseDate"],
                "descriptions": digimon["descriptions"],
                "xAntibody": digimon["xAntibody"],
                "priorEvolutions": extract_field_values(digimon, "priorEvolutions"),
                "nextEvolutions": extract_field_values(digimon, "nextEvolutions"),
                "levels": extract_field_values(digimon, "levels"),
                "types": extract_field_values(digimon, "types"),
                "attributes": extract_field_values(digimon, "attributes"),
                "skills": extract_field_values(digimon, "skills"),
                "fields": extract_field_values(digimon, "fields"),
            })
            if digimon_list.index(digimon) >= 5:  # Print the first 5 digimons for verification
                break;
    
        df_digimons = pd.DataFrame(digimons)

        output_path = Path(f"data/processed/digimons.csv")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df_digimons.to_csv("data/processed/digimons.csv", index=False)



        
    except Exception as e:
        print(f"Error: {e}")

In [37]:
transform_data()

In [ ]:


# Requer: pip install psycopg2-binary

def load_json_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo nao encontrado: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def parse_release_date(value):
    if value is None or value == "":
        return None

    # inteiro (ex.: 1997)
    if isinstance(value, int):
        return date(value, 1, 1)

    # string
    if isinstance(value, str):
        v = value.strip()

        # ano simples (ex.: "1997")
        if len(v) == 4 and v.isdigit():
            return date(int(v), 1, 1)

        # tenta ISO (YYYY-MM-DD)
        try:
            return datetime.fromisoformat(v).date()
        except ValueError:
            return None

    return None

def upsert_lookup_table(cur, table_name, records):
    # Tabelas: attributes, fields, levels, skills, types
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id BIGINT PRIMARY KEY,
            name TEXT,
            payload JSONB
        );
    """)

    for r in records or []:
        cur.execute(
            f"""
            INSERT INTO {table_name} (id, name, payload)
            VALUES (%s, %s, %s)
            ON CONFLICT (id) DO UPDATE
            SET name = EXCLUDED.name,
                payload = EXCLUDED.payload;
            """,
            (r.get("id"), r.get("name"), Json(r))
        )

def create_schema(cur):
    cur.execute("""
        CREATE TABLE IF NOT EXISTS digimons (
            id BIGINT PRIMARY KEY,
            name TEXT NOT NULL,
            release_date DATE NULL,
            x_antibody BOOLEAN,
            images JSONB,
            descriptions JSONB
        );

        CREATE TABLE IF NOT EXISTS digimon_levels (
            digimon_id BIGINT NOT NULL,
            level_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, level_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (level_id) REFERENCES levels(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_types (
            digimon_id BIGINT NOT NULL,
            type_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, type_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (type_id) REFERENCES types(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_attributes (
            digimon_id BIGINT NOT NULL,
            attribute_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, attribute_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (attribute_id) REFERENCES attributes(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_fields (
            digimon_id BIGINT NOT NULL,
            field_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, field_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (field_id) REFERENCES fields(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_skills (
            digimon_id BIGINT NOT NULL,
            skill_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, skill_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (skill_id) REFERENCES skills(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_prior_evolutions (
            digimon_id BIGINT NOT NULL,
            prior_digimon_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, prior_digimon_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (prior_digimon_id) REFERENCES digimons(id) ON DELETE RESTRICT
        );

        CREATE TABLE IF NOT EXISTS digimon_next_evolutions (
            digimon_id BIGINT NOT NULL,
            next_digimon_id BIGINT NOT NULL,
            PRIMARY KEY (digimon_id, next_digimon_id),
            FOREIGN KEY (digimon_id) REFERENCES digimons(id) ON DELETE CASCADE,
            FOREIGN KEY (next_digimon_id) REFERENCES digimons(id) ON DELETE RESTRICT
        );
    """)

def insert_relation_if_exists(cur, table_name, left_col, right_col, right_table, left_id, right_id):
    cur.execute(
        f"""
        INSERT INTO {table_name} ({left_col}, {right_col})
        SELECT %s, %s
        WHERE EXISTS (SELECT 1 FROM {right_table} WHERE id = %s)
        ON CONFLICT DO NOTHING;
        """,
        (left_id, right_id, right_id),
    )

def insert_relations(cur, digimon):
    digimon_id = digimon["id"]

    for level_id in extract_field_values(digimon, "levels"):
        insert_relation_if_exists(cur, "digimon_levels", "digimon_id", "level_id", "levels", digimon_id, level_id)

    for type_id in extract_field_values(digimon, "types"):
        insert_relation_if_exists(cur, "digimon_types", "digimon_id", "type_id", "types", digimon_id, type_id)

    for attribute_id in extract_field_values(digimon, "attributes"):
        insert_relation_if_exists(cur, "digimon_attributes", "digimon_id", "attribute_id", "attributes", digimon_id, attribute_id)

    for field_id in extract_field_values(digimon, "fields"):
        insert_relation_if_exists(cur, "digimon_fields", "digimon_id", "field_id", "fields", digimon_id, field_id)

    for skill_id in extract_field_values(digimon, "skills"):
        insert_relation_if_exists(cur, "digimon_skills", "digimon_id", "skill_id", "skills", digimon_id, skill_id)

    for prior_id in extract_field_values(digimon, "priorEvolutions"):
        insert_relation_if_exists(cur, "digimon_prior_evolutions", "digimon_id", "prior_digimon_id", "digimons", digimon_id, prior_id)

    for next_id in extract_field_values(digimon, "nextEvolutions"):
        insert_relation_if_exists(cur, "digimon_next_evolutions", "digimon_id", "next_digimon_id", "digimons", digimon_id, next_id)

def get_db_password():
    password = os.getenv("PGPASSWORD")
    if password:
        return password

    print("PGPASSWORD nao definida. Informe a senha do PostgreSQL:")
    password = getpass("Senha: ").strip()
    if not password:
        raise ValueError("Senha vazia. Defina PGPASSWORD ou informe uma senha valida.")
    return password

def get_db_config():
    return {
        "host": os.getenv("PGHOST", "localhost"),
        "port": os.getenv("PGPORT", "5432"),
        "dbname": os.getenv("PGDATABASE", "digimon_db"),
        "user": os.getenv("PGUSER", "postgres"),
        "password": get_db_password(),
    }

def ensure_database_exists(db_cfg):
    maintenance_db = os.getenv("PGMAINTENANCE_DB", "postgres")
    admin_conn = psycopg2.connect(
        host=db_cfg["host"],
        port=db_cfg["port"],
        dbname=maintenance_db,
        user=db_cfg["user"],
        password=db_cfg["password"],
    )

    try:
        admin_conn.autocommit = True
        with admin_conn.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (db_cfg["dbname"],))
            if cur.fetchone() is None:
                cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_cfg["dbname"])))
                print(f"Banco {db_cfg['dbname']} criado com sucesso.")
    finally:
        admin_conn.close()

def load_to_postgres():
    db_cfg = get_db_config()
    ensure_database_exists(db_cfg)

    conn = psycopg2.connect(**db_cfg)
    conn.autocommit = False

    try:
        attributes = load_json_file("data/raw/attributes.json")
        fields_data = load_json_file("data/raw/fields.json")
        levels = load_json_file("data/raw/levels.json")
        skills = load_json_file("data/raw/skills.json")

        types_path = "data/raw/types.json"
        if os.path.exists(types_path):
            types = load_json_file(types_path)
        else:
            print("Aviso: data/raw/types.json nao encontrado. Relacao digimon_types sera carregada apenas se o arquivo existir.")
            types = []

        digimons = load_json_file("data/raw/digimons.json")

        with conn.cursor() as cur:
            upsert_lookup_table(cur, "attributes", attributes)
            upsert_lookup_table(cur, "fields", fields_data)
            upsert_lookup_table(cur, "levels", levels)
            upsert_lookup_table(cur, "skills", skills)
            upsert_lookup_table(cur, "types", types)

            create_schema(cur)

            for d in digimons:
                cur.execute(
                    """
                    INSERT INTO digimons (id, name, release_date, x_antibody, images, descriptions)
                    VALUES (%s, %s, %s, %s, %s, %s)
                    ON CONFLICT (id) DO UPDATE
                    SET name = EXCLUDED.name,
                        release_date = EXCLUDED.release_date,
                        x_antibody = EXCLUDED.x_antibody,
                        images = EXCLUDED.images,
                        descriptions = EXCLUDED.descriptions;
                    """,
                    (
                        d.get("id"),
                        d.get("name"),
                        parse_release_date(d.get("releaseDate")),
                        d.get("xAntibody"),
                        Json(d.get("images", [])),
                        Json(d.get("descriptions", [])),
                    ),
                )

            for d in digimons:
                insert_relations(cur, d)

        conn.commit()
        print("Carga finalizada com sucesso no PostgreSQL.")
    except Exception as e:
        conn.rollback()
        print(f"Erro ao carregar dados: {e}")
        raise
    finally:
        conn.close()

load_to_postgres()

PGPASSWORD nao definida. Informe a senha do PostgreSQL:


Banco digimon_db criado com sucesso.
Carga finalizada com sucesso no PostgreSQL.


In [51]:
# Validacao: d.* mostra apenas colunas da tabela digimons.
# Para ver relacionamentos, use JOINs/tabelas ponte.

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "digimon_db"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "7904")
)

try:
    counts_sql = """
    SELECT 'digimons' AS tabela, COUNT(*) AS total FROM digimons
    UNION ALL SELECT 'digimon_levels', COUNT(*) FROM digimon_levels
    UNION ALL SELECT 'digimon_types', COUNT(*) FROM digimon_types
    UNION ALL SELECT 'digimon_attributes', COUNT(*) FROM digimon_attributes
    UNION ALL SELECT 'digimon_fields', COUNT(*) FROM digimon_fields
    UNION ALL SELECT 'digimon_skills', COUNT(*) FROM digimon_skills
    UNION ALL SELECT 'digimon_prior_evolutions', COUNT(*) FROM digimon_prior_evolutions
    UNION ALL SELECT 'digimon_next_evolutions', COUNT(*) FROM digimon_next_evolutions
    ORDER BY tabela;
    """
    print("Contagem das tabelas:")
    display(pd.read_sql_query(counts_sql, conn))

    relations_sql = """
    SELECT
        d.id,
        d.name,
        d.release_date,
        d.x_antibody,
        d.images,
        d.descriptions,
        COALESCE(array_agg(DISTINCT l.name) FILTER (WHERE l.id IS NOT NULL), '{}') AS levels,
        COALESCE(array_agg(DISTINCT t.name) FILTER (WHERE t.id IS NOT NULL), '{}') AS types,
        COALESCE(array_agg(DISTINCT a.name) FILTER (WHERE a.id IS NOT NULL), '{}') AS attributes,
        COALESCE(array_agg(DISTINCT f.name) FILTER (WHERE f.id IS NOT NULL), '{}') AS fields,
        COALESCE(array_agg(DISTINCT s.name) FILTER (WHERE s.id IS NOT NULL), '{}') AS skills,
        COALESCE(array_agg(DISTINCT p.name) FILTER (WHERE p.id IS NOT NULL), '{}') AS prior_evolutions,
        COALESCE(array_agg(DISTINCT n.name) FILTER (WHERE n.id IS NOT NULL), '{}') AS next_evolutions
    FROM digimons d
    LEFT JOIN digimon_levels dl ON dl.digimon_id = d.id
    LEFT JOIN levels l ON l.id = dl.level_id
    LEFT JOIN digimon_types dt ON dt.digimon_id = d.id
    LEFT JOIN types t ON t.id = dt.type_id
    LEFT JOIN digimon_attributes da ON da.digimon_id = d.id
    LEFT JOIN attributes a ON a.id = da.attribute_id
    LEFT JOIN digimon_fields df ON df.digimon_id = d.id
    LEFT JOIN fields f ON f.id = df.field_id
    LEFT JOIN digimon_skills ds ON ds.digimon_id = d.id
    LEFT JOIN skills s ON s.id = ds.skill_id
    LEFT JOIN digimon_prior_evolutions dpe ON dpe.digimon_id = d.id
    LEFT JOIN digimons p ON p.id = dpe.prior_digimon_id
    LEFT JOIN digimon_next_evolutions dne ON dne.digimon_id = d.id
    LEFT JOIN digimons n ON n.id = dne.next_digimon_id
    GROUP BY d.id, d.name, d.release_date, d.x_antibody, d.images, d.descriptions
    ORDER BY d.id
    LIMIT 20;
    """
    print("Amostra com relacionamentos:")
    display(pd.read_sql_query(relations_sql, conn))

finally:
    conn.close()

Contagem das tabelas:


C:\Users\Lucas.BUSTERSTARK\AppData\Local\Temp\ipykernel_24660\3950605293.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql_query(counts_sql, conn))


,tabela,total
0,digimon_attributes,1645
1,digimon_fields,1636
2,digimon_levels,1586
3,digimon_next_evolutions,15551
4,digimon_prior_evolutions,15550
5,digimon_skills,4673
6,digimon_types,1436
7,digimons,1488


Amostra com relacionamentos:


C:\Users\Lucas.BUSTERSTARK\AppData\Local\Temp\ipykernel_24660\3950605293.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql_query(relations_sql, conn))


,id,name,release_date,x_antibody,images,descriptions,levels,types,attributes,fields,skills,prior_evolutions,next_evolutions
0,1,Agumon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Child],[Reptile],[Vaccine],"[Deep Savers, Dragon's Roar, Metal Empire, Nat...","[Aerial Mach Jab, Agumon Dive, Baby Burner, Ba...","[Algomon (Baby II), Gigimon, Greymon, Koromon,...","[Agnimon, Agumon -Yuki no Kizuna-, Agumon (Bla..."
1,2,Airdramon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Adult],[Mythical Beast],[Vaccine],"[Dragon's Roar, Nature Spirits, Wind Guardians]","[Big Jaw, God Tornado, Megalo Spark, Spinning ...","[Agumon, Agumon (2006 Anime Version), Agumon (...","[Aero V-dramon, Andiramon, Andiramon (Deva), A..."
2,3,Angemon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Adult],[Angel],[Vaccine],"[Virus Busters, Wind Guardians]","[Angel Slam, Glide, God Typhoon, Halo Attack, ...","[Agumon, Agumon (2006 Anime Version), Agumon (...","[Aero V-dramon, Agumon (Black) (2006 Anime Ver..."
3,4,Betamon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Child],[Amphibian],[Virus],[Nature Spirits],"[Beta Slugger, Cutter Fin, Dengeki Biririn, Wa...","[Budmon, Chapmon, Chicomon, Koromon, Pukamon, ...","[Airdramon, Betamon (X-Antibody), Bitmon, Coel..."
4,5,Birdramon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Adult],[Giant Bird],"[Data, Vaccine]","[Nature Spirits, Wind Guardians]","[Bir-Flame, Fire Flap, Fireball, Kyōushū, Mach...","[Agumon, Agumon (2006 Anime Version), Agumon (...","[Aero V-dramon, Agnimon, Airdramon, Andromon, ..."
5,6,Botamon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'reference_book', 'language': 'jap...",[Baby I],[Slime],"[Data, Free]","[Nature Spirits, Virus Busters]","[Awa, San no Awa]",[],"[Agumon (Black), Airdramon, Budmon, Candmon, K..."
6,7,Bun,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'fanmade', 'language': 'jap', 'des...",[Child],[],[],[],[Spinning Tail Cyclone],[],[]
7,8,Death Airdramon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,[],[],[Mythical Beast],[],[],[Death Arrow],[Deathmon (C'mon Digimon Version)],"[Death Devimon, Deathmon (C'mon Digimon Version)]"
8,9,Death Devimon,1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'fanmade', 'language': 'jap', 'des...",[],[],[],[],[Death Knuckle],"[Death Airdramon, Deathmon (C'mon Digimon Vers...","[Death Tyranomon, Deathmon (C'mon Digimon Vers..."
9,10,Death Meramon (C'mon Digimon Version),1997-01-01,False,[{'href': 'https://digi-api.com/images/digimon...,"[{'origin': 'fanmade', 'language': 'jap', 'des...",[],[],[],[],[],"[Death Metal Greymon, Deathmon (C'mon Digimon ...",[Deathmon (C'mon Digimon Version)]
